<a href="https://colab.research.google.com/github/KCaroca/An-lisis-y-Predicci-n-de-Ventas-en-una-Tienda-de-Retail-Core-/blob/development/Python_para_Ciencia_de_Datos_Proyecto_I_An%C3%A1lisis_y_predicci%C3%B3n_de_ventas_Part_1_(Core).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análisis y Predicción de Ventas en una Tienda de Retail (Core)


# Instrucciones
**Exploración de Datos:**
Calcula el total de ventas por categoría de producto.
Calcula el promedio de ventas diarias por categoría de producto.
Identifica las categorías de productos con mayores y menores ventas.
**Manipulación de Datos:**
Filtra los datos para mostrar solo las ventas de una categoría de producto específica.
Realiza operaciones de suma, resta, multiplicación y división en los datos para obtener estadísticas adicionales.

# Importar Numpy y Cargar Datos



In [9]:
import numpy as np

def cargar_datos(ruta_archivo):
    # Carga los datos del archivo CSV utilizando NumPy
    datos = np.genfromtxt(ruta_archivo, delimiter=',', skip_header=1, dtype=str) # Added dtype=str for robustness
    return datos

if __name__ == "__main__":
    ruta_archivo = '/content/drive/MyDrive/retail_sales_dataset.csv' # Corrected file path
    datos = cargar_datos(ruta_archivo)
    print(datos)

[['1' '2023-11-24' 'CUST001' ... '3' '50' '150']
 ['2' '2023-02-27' 'CUST002' ... '2' '500' '1000']
 ['3' '2023-01-13' 'CUST003' ... '1' '30' '30']
 ...
 ['998' '2023-10-29' 'CUST998' ... '4' '25' '100']
 ['999' '2023-12-05' 'CUST999' ... '3' '50' '150']
 ['1000' '2023-04-12' 'CUST1000' ... '4' '30' '120']]


# Exploración de Datos


In [21]:
# Cargar los nombres de las columnas desde el archivo CSV
with open(ruta_archivo, 'r') as f:
    header = f.readline().strip().split(',')

print("Nombres de las columnas:")
print(header)
print("\nPrimeras 5 filas de datos (sin encabezado):")
# Como `datos` ya está cargado sin el encabezado, solo mostramos las primeras filas
for row in datos[:5]:
    print(row)


Nombres de las columnas:
['Transaction ID', 'Date', 'Customer ID', 'Gender', 'Age', 'Product Category', 'Quantity', 'Price per Unit', 'Total Amount']

Primeras 5 filas de datos (sin encabezado):
['1' '2023-11-24' 'CUST001' 'Male' '34' 'Beauty' '3' '50' '150']
['2' '2023-02-27' 'CUST002' 'Female' '26' 'Clothing' '2' '500' '1000']
['3' '2023-01-13' 'CUST003' 'Male' '50' 'Electronics' '1' '30' '30']
['4' '2023-05-21' 'CUST004' 'Male' '37' 'Clothing' '1' '500' '500']
['5' '2023-05-06' 'CUST005' 'Male' '30' 'Beauty' '2' '50' '100']


In [3]:
print(datos.shape)

(1000, 9)


## **Calcular el total de ventas por categoría de producto.**

In [8]:
categoria_producto = datos[:,5]
Fecha = datos[:,1]
Total_venta = datos[:,8].astype(float)
Cantidad = datos[:,6].astype(float)
Precio_unitario = datos[:,7].astype(float)

categorias_unicas= np.unique(categoria_producto)
print("categoria_producto:",categorias_unicas)

total_ventas_por_categoría = {}
for categoria in categorias_unicas:
    #Filtra las ventas totales para la categoría actual
    ventas_por_categoría = Total_venta[categoria_producto == categoria]
    # Suma las ventas para obtener el total de la categoría
    total_ventas_por_categoría[categoria] = np.sum(ventas_por_categoría)

print("Total de ventas por categoría:")
for categoria, total_ventas in total_ventas_por_categoría.items():
    print(f"- {categoria}: {total_ventas:.0f}")

categoria_producto: ['Beauty' 'Clothing' 'Electronics']
Total de ventas por categoría:
- Beauty: 143515
- Clothing: 155580
- Electronics: 156905


# **Calcula el promedio de ventas diarias por categoría de producto.**





In [12]:
# Enfoque de múltiples pasos.

# 1. Crear una clave combinada de Fecha y Categoría para identificar ventas diarias por categoría única.
#    Usamos una concatenación de cadenas como clave.

combined_daily_category_keys = np.array([f"{d}_{c}" for d, c in zip(Fecha, categoria_producto)])

# 2. Encontrar las combinaciones únicas de (Fecha, Categoría) y los índices inversos.
#    Los índices inversos nos dirán a qué combinación única pertenece cada venta original.

unique_daily_category_combinations, inverse_indices = np.unique(combined_daily_category_keys, return_inverse=True)

# 3. Sumar el 'Total Amount' para cada combinación única de Fecha y Categoría.
#    Esto nos da las ventas totales para cada día y categoría específica.

daily_category_sales_sums = np.zeros(len(unique_daily_category_combinations), dtype=float)
np.add.at(daily_category_sales_sums, inverse_indices, Total_venta)

# 4. Extraer solo la categoría de producto de las claves combinadas únicas.
#    Por ejemplo, de '2023-01-01_Beauty' obtenemos 'Beauty'.
categories_from_unique_daily_sums = np.array([key.split('_')[1] for key in unique_daily_category_combinations])

# 5. Agrupar estas sumas diarias por la categoría de producto.
#    Esto es para preparar el cálculo del promedio final por categoría.

#    Obtener las categorías de producto únicas para la agrupación final.
final_unique_categories, final_inverse_indices = np.unique(categories_from_unique_daily_sums, return_inverse=True)
#    Sumar todas las 'sumas de ventas diarias por categoría' para cada categoría de producto.
#    Por ejemplo, para 'Beauty', sumamos todas sus ventas de '2023-01-01_Beauty', '2023-01-02_Beauty', etc.
sum_of_all_daily_sales_per_category = np.zeros(len(final_unique_categories), dtype=float)
np.add.at(sum_of_all_daily_sales_per_category, final_inverse_indices, daily_category_sales_sums)

#    Contar cuántas 'sumas de ventas diarias' contribuyeron a cada categoría.
#    Esto es el denominador para el promedio.
count_of_daily_records_per_category = np.zeros(len(final_unique_categories), dtype=int)
np.add.at(count_of_daily_records_per_category, final_inverse_indices, 1)

# 6. Calcular el promedio de ventas diarias por categoría.
#    Manejamos la división por cero si alguna categoría no tuvo registros diarios.
average_daily_sales_per_category_numpy = np.divide(
    sum_of_all_daily_sales_per_category,
    count_of_daily_records_per_category,
    out=np.full(len(final_unique_categories), np.nan), # Asigna NaN si el denominador es cero
    where=count_of_daily_records_per_category != 0
)

print("Promedio de ventas diarias por categoría de producto (usando NumPy):")
for i, category in enumerate(final_unique_categories):
    print(f"- {category}: {average_daily_sales_per_category_numpy[i]:.2f}")

Promedio de ventas diarias por categoría de producto (usando NumPy):
- Beauty: 703.50
- Clothing: 670.60
- Electronics: 716.46


# **Categorías de productos con mayores y menores ventas.**

In [17]:
# Encontrar la categoría con MAYOR y MENOR venta total

# max() busca la clave (categoría) que tiene el valor más alto en el diccionario
max_categoria = max(total_ventas_por_categoría, key=total_ventas_por_categoría.get)

# min() busca la clave (categoría) que tiene el valor más bajo en el diccionario
min_categoria = min(total_ventas_por_categoría, key=total_ventas_por_categoría.get)

# Mostramos la categoría con mayor venta junto con su valor total
print("Categoría con MAYOR venta:")
print(f"{max_categoria}: {total_ventas_por_categoría[max_categoria]:.0f}")

# Mostramos la categoría con menor venta junto con su valor total
print("\nCategoría con MENOR venta:")
print(f"{min_categoria}: {total_ventas_por_categoría[min_categoria]:.0f}")

Categoría con MAYOR venta:
Electronics: 156905

Categoría con MENOR venta:
Beauty: 143515
